In [ ]:
#!/usr/bin/env python3


In [ ]:
# =============================================================================
# HierAttn-Block — HDFS Log Anomaly Detection  (Kaggle / Local)
#
# Steps:
#   1  Environment Setup
#   2  Session Construction
#   3  Feature Extraction
#   4  PyTorch Dataset & DataLoaders
#   5  Baseline 1: DeepLog
#   6  Baseline 2: LogBERT (simplified)
#   7  Main Model: HierAttn-Block
#   8  Training
#   9  Two-Stage Inference
#  10  Ablation Study
#  11  Thesis Figures (PNG 300 DPI)
#      Final Summary
#
# Dataset: HDFS_Drain.csv  (log, label, template columns)


In [ ]:
# =============================================================================

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1 — ENVIRONMENT SETUP                                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, re, math, random, time, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from collections import Counter, defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
)
from sklearn.preprocessing import label_binarize
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier

# ── Random seed ───────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {DEVICE}")

# ── Paths ─────────────────────────────────────────────────────────────────────
KAGGLE = os.path.exists("/kaggle/working")
if KAGGLE:
    DATA_DIR   = "/kaggle/input/logs-drain-datasets-hdfs-bgl-spirit"
    OUTPUT_DIR = "/kaggle/working/hierattn_output"
else:
    DATA_DIR   = "./Dataset"
    OUTPUT_DIR = "result/results_DeepLogEnhanced_HDFS_v2/hierattn_output"

FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")
MODEL_DIR  = os.path.join(OUTPUT_DIR, "models")
for d in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

CSV_PATH = os.path.join(DATA_DIR, "HDFS_Drain.csv")

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LEN    = 32      # max events per session (padded/truncated)
N_ROWS     = 500_000 # rows to read from CSV (None = all)
BATCH_SIZE = 128
MAX_EPOCHS = 50
LR         = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE   = 7       # early stopping patience

# ── Load CSV ──────────────────────────────────────────────────────────────────
print(f"\n📁 Loading {CSV_PATH} ...")
df_raw = pd.read_csv(CSV_PATH, nrows=N_ROWS, on_bad_lines="skip", low_memory=False)
print(f"   Raw shape: {df_raw.shape}")

# Normalise column names
df_raw.columns = [c.lower().strip() for c in df_raw.columns]
# rename 'label' variants
for src, tgt in [("Label","label"),("Template","template"),("Log","log"),("Content","log"),("content","log")]:
    if src in df_raw.columns and tgt not in df_raw.columns:
        df_raw.rename(columns={src: tgt}, inplace=True)

print(f"   Columns: {list(df_raw.columns)}")
print(f"\n📊 Label distribution:")
print(df_raw["label"].value_counts())

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2 — SESSION CONSTRUCTION                                              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 2 — SESSION CONSTRUCTION")
print("="*60)

# ── Extract Block ID ──────────────────────────────────────────────────────────
df_raw["block_id"] = df_raw["log"].str.extract(r"(blk_-?\d+)")
df_raw = df_raw.dropna(subset=["block_id"]).copy()
print(f"   Rows with block_id: {len(df_raw):,}")

# ── Encode labels ─────────────────────────────────────────────────────────────
df_raw["is_anomaly"] = (df_raw["label"].str.strip() != "Normal").astype(int)

# ── Template → integer ID ─────────────────────────────────────────────────────
unique_templates = df_raw["template"].dropna().unique().tolist()
template2id = {"<PAD>": 0, "<UNK>": 1}
for i, t in enumerate(sorted(unique_templates)):
    template2id[t] = i + 2
VOCAB_SIZE = len(template2id)
df_raw["event_id"] = df_raw["template"].map(template2id).fillna(1).astype(int)

print(f"   Vocabulary size: {VOCAB_SIZE} templates")

# ── Parse numeric fields from log ─────────────────────────────────────────────
# size (bytes), IP address, thread ID — best effort regex
def parse_log_fields(log_str):
    """Return (size_bytes, ip_hash, thread_id) from a raw log line."""
    # Size: look for numbers followed by (B|KB|MB|GB|bytes)
    size = 0
    m = re.search(r"(\d+)\s*(?:bytes?|B\b)", log_str, re.I)
    if m:
        size = int(m.group(1))
    else:
        # fallback: extract last 6-digit+ number that could be a size
        nums = re.findall(r"\b(\d{6,})\b", log_str)
        if nums:
            size = int(nums[-1])

    # IP: first IPv4 pattern
    ip = 0
    m_ip = re.search(r"(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})", log_str)
    if m_ip:
        octs = [int(x) for x in m_ip.groups()]
        ip = (octs[0]*256**3 + octs[1]*256**2 + octs[2]*256 + octs[3]) % 1000

    # Thread: look for thread/tid followed by number
    thread = 0
    m_t = re.search(r"(?:thread|tid)[:\s]*(\d+)", log_str, re.I)
    if m_t:
        thread = int(m_t.group(1))
    else:
        m_t2 = re.search(r"\b(\d{4,6})\b", log_str)
        if m_t2:
            thread = int(m_t2.group(1))

    return size, ip, thread

print("   Parsing log fields (size, ip, thread) ...")
parsed = df_raw["log"].apply(parse_log_fields)
df_raw["size_val"]   = parsed.apply(lambda x: x[0])
df_raw["ip_hash"]    = parsed.apply(lambda x: x[1])
df_raw["thread_id"]  = parsed.apply(lambda x: x[2])

# Timestamp — use row index as proxy for ordering within a block
df_raw["row_idx"] = np.arange(len(df_raw))

# ── Group into sessions ────────────────────────────────────────────────────────
print("   Grouping rows into sessions by block_id ...")

def agg_session(grp):
    return pd.Series({
        "event_ids":    list(grp["event_id"]),
        "templates":    list(grp["template"].fillna("<UNK>")),
        "size_vals":    list(grp["size_val"]),
        "ip_hashes":    list(grp["ip_hash"]),
        "thread_ids":   list(grp["thread_id"]),
        "row_indices":  list(grp["row_idx"]),
        "label":        int(grp["is_anomaly"].max()),
    })

sessions = df_raw.groupby("block_id").apply(agg_session).reset_index()

n_total   = len(sessions)
n_normal  = (sessions["label"] == 0).sum()
n_anomaly = (sessions["label"] == 1).sum()

print(f"\n   Total sessions : {n_total:,}")
print(f"   Normal         : {n_normal:,}  ({100*n_normal/n_total:.1f}%)")
print(f"   Anomaly        : {n_anomaly:,}  ({100*n_anomaly/n_total:.1f}%)")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 — FEATURE EXTRACTION                                                ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 3 — FEATURE EXTRACTION")
print("="*60)

# ── Identify top-5 template IDs (most common) for structural features ──────────
# Use template frequency to define the "main 5" template types
template_freq = Counter(df_raw["template"].tolist())
top5_templates = [t for t, _ in template_freq.most_common(5)]
top5_ids       = [template2id.get(t, 1) for t in top5_templates]
print(f"   Top-5 template IDs (for structural feat): {top5_ids}")

# ── Normalisation stats for thread_id ─────────────────────────────────────────
thread_max = df_raw["thread_id"].max()
if thread_max == 0:
    thread_max = 1

# ── Sinusoidal positional encoding helper ─────────────────────────────────────
def sinusoidal_encoding(positions, d_model=32):
    """Encode a 1-D array of scalar positions into d_model-dim sinusoidal vectors."""
    pe = np.zeros((len(positions), d_model), dtype=np.float32)
    for i, pos in enumerate(positions):
        for k in range(0, d_model, 2):
            denom = 10000 ** (k / d_model)
            pe[i, k]   = math.sin(pos / denom)
            if k + 1 < d_model:
                pe[i, k+1] = math.cos(pos / denom)
    return pe

def extract_features(row):
    """Return all tensors for one session."""
    evs   = row["event_ids"]
    sizes = row["size_vals"]
    ips   = row["ip_hashes"]
    thrs  = row["thread_ids"]
    ridxs = row["row_indices"]
    n     = len(evs)

    # ── A) Sequence features (padded to MAX_LEN) ──────────────────────────────
    seq_len = min(n, MAX_LEN)

    # event_ids
    event_ids = np.zeros(MAX_LEN, dtype=np.int64)
    event_ids[:seq_len] = evs[:seq_len]

    # param_feats: [log(size+1), ip_hash mod 1000, thread_id normalised]
    param_feats = np.zeros((MAX_LEN, 3), dtype=np.float32)
    for i in range(seq_len):
        param_feats[i, 0] = math.log(sizes[i] + 1)
        param_feats[i, 1] = (ips[i] % 1000) / 1000.0
        param_feats[i, 2] = min(thrs[i], thread_max) / thread_max

    # time_deltas: use row-index difference as proxy for "seconds since session start"
    time_deltas = np.zeros(MAX_LEN, dtype=np.float32)
    base_idx = ridxs[0]
    for i in range(seq_len):
        time_deltas[i] = float(ridxs[i] - base_idx)

    # sinusoidal encoding of time_deltas
    sinusoidal_td = sinusoidal_encoding(time_deltas[:seq_len], d_model=32)  # (seq_len, 32)
    # Pad
    sin_padded = np.zeros((MAX_LEN, 32), dtype=np.float32)
    sin_padded[:seq_len] = sinusoidal_td

    # attention_mask
    attention_mask = np.zeros(MAX_LEN, dtype=np.float32)
    attention_mask[:seq_len] = 1.0

    # ── B) Structural features (11-dim) ──────────────────────────────────────
    # 1-5: count of each of 5 main template types
    ev_arr = np.array(evs)
    struct = np.zeros(11, dtype=np.float32)
    for k, tid in enumerate(top5_ids):
        struct[k] = float(np.sum(ev_arr == tid))

    # 6: std of sizes (0 if missing)
    struct[5] = float(np.std(sizes)) if len(sizes) > 1 else 0.0

    # 7: number of unique IPs
    struct[6] = float(len(set(ips)))

    # 8: session duration (row-index span as proxy)
    struct[7] = float(ridxs[-1] - ridxs[0]) if n > 1 else 0.0

    # 9: max inter-event gap
    if n > 1:
        gaps = [ridxs[i+1] - ridxs[i] for i in range(n-1)]
        struct[8] = float(max(gaps))
    else:
        struct[8] = 0.0

    # 10: missing_allocate (1 if template "blkAllocate" type not seen)
    # We check if template containing "allocate" is absent
    allocate_id = None
    for t, tid in template2id.items():
        if "allocat" in t.lower():
            allocate_id = tid
            break
    if allocate_id is not None:
        struct[9] = 0.0 if (allocate_id in ev_arr) else 1.0
    else:
        struct[9] = 0.0  # can't determine → default 0

    # 11: replication_count != 3
    # Check for template mentioning "replication" — count occurrences
    repl_count = 0
    for t in row["templates"]:
        if "replicat" in t.lower():
            repl_count += 1
    struct[10] = 1.0 if (repl_count != 3) else 0.0

    return {
        "event_ids":     event_ids,
        "param_feats":   param_feats,
        "time_deltas":   time_deltas,          # raw scalar (for reference)
        "sin_time":      sin_padded,           # sinusoidal encoded
        "struct_feats":  struct,
        "attention_mask": attention_mask,
        "label":         int(row["label"]),
        "repl_count":    float(repl_count),
        "missing_alloc": int(struct[9]),
        "repl_neq3":     int(struct[10]),
    }

print("   Extracting features for all sessions ...")
features = [extract_features(sessions.iloc[i]) for i in range(len(sessions))]
print(f"   Done: {len(features)} feature dicts")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 4 — PYTORCH DATASET AND DATALOADERS                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 4 — PYTORCH DATASET AND DATALOADERS")
print("="*60)

class HDFSDataset(Dataset):
    """Returns all feature tensors for one HDFS session."""

    def __init__(self, feat_list):
        self.data = feat_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]
        return (
            torch.tensor(d["event_ids"],      dtype=torch.long),     # (32,)
            torch.tensor(d["param_feats"],    dtype=torch.float32),  # (32, 3)
            torch.tensor(d["sin_time"],       dtype=torch.float32),  # (32, 32)
            torch.tensor(d["struct_feats"],   dtype=torch.float32),  # (11,)
            torch.tensor(d["attention_mask"], dtype=torch.float32),  # (32,)
            torch.tensor(d["label"],          dtype=torch.long),     # scalar
            torch.tensor(d["repl_count"],     dtype=torch.float32),  # scalar
        )

# ── Stratified 70/10/20 split ─────────────────────────────────────────────────
labels_all = np.array([f["label"] for f in features])
indices    = np.arange(len(features))

# 80/20 → then 70/10 from the 80
idx_tv, idx_test = train_test_split(
    indices, test_size=0.20, stratify=labels_all, random_state=SEED
)
labels_tv = labels_all[idx_tv]
idx_train, idx_val = train_test_split(
    idx_tv, test_size=0.10/0.80, stratify=labels_tv, random_state=SEED
)

feat_train = [features[i] for i in idx_train]
feat_val   = [features[i] for i in idx_val]
feat_test  = [features[i] for i in idx_test]

print(f"   Train: {len(feat_train):,} | Val: {len(feat_val):,} | Test: {len(feat_test):,}")
for name, feat_list in [("Train", feat_train), ("Val", feat_val), ("Test", feat_test)]:
    lbl = np.array([f["label"] for f in feat_list])
    print(f"   {name} → Normal: {(lbl==0).sum():,}  Anomaly: {(lbl==1).sum():,}  "
          f"({100*lbl.mean():.1f}% anomaly)")

ds_train = HDFSDataset(feat_train)
ds_val   = HDFSDataset(feat_val)
ds_test  = HDFSDataset(feat_test)

# Weighted sampler to handle class imbalance in training
train_labels = np.array([f["label"] for f in feat_train])
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(feat_train),
    replacement=True
)

dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=0, pin_memory=False)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)
dl_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,    num_workers=0)

print(f"\n   DataLoaders ready. Batches per epoch (train): {len(dl_train)}")

# ── Also prepare flat event-ID arrays for baselines ──────────────────────────
X_train_seq = np.array([f["event_ids"] for f in feat_train])
X_val_seq   = np.array([f["event_ids"] for f in feat_val])
X_test_seq  = np.array([f["event_ids"] for f in feat_test])
y_train     = np.array([f["label"] for f in feat_train])
y_val       = np.array([f["label"] for f in feat_val])
y_test      = np.array([f["label"] for f in feat_test])

# ── Structural arrays for ablation and sklearn baselines ──────────────────────
X_train_struct = np.array([f["struct_feats"] for f in feat_train])
X_val_struct   = np.array([f["struct_feats"] for f in feat_val])
X_test_struct  = np.array([f["struct_feats"] for f in feat_test])

# ── Hard-rule arrays for two-stage inference ──────────────────────────────────
test_missing_alloc = np.array([f["missing_alloc"] for f in feat_test])
test_repl_neq3     = np.array([f["repl_neq3"]     for f in feat_test])

# ── Tracking dict ─────────────────────────────────────────────────────────────
baseline_results = {}
results          = {}
all_roc          = {}   # {model_name: (fpr, tpr, auc)}

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 5 — BASELINE 1: DeepLog                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 5 — BASELINE 1: DeepLog")
print("="*60)

class DeepLogLSTM(nn.Module):
    """DeepLog (Du et al. 2017) — unidirectional 2-layer LSTM next-event predictor."""

    def __init__(self, vocab_size, hidden=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, hidden, num_layers,
                            batch_first=True, dropout=0.3)
        self.head = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)          # (B, L, 64)
        out, _ = self.lstm(emb)          # (B, L, H)
        logits = self.head(out)          # (B, L, V)
        return logits

# ── Training on normal sessions only (unsupervised as per paper) ──────────────
normal_mask = y_train == 0
X_dl_normal = torch.from_numpy(X_train_seq[normal_mask]).long()

class WindowDataset(Dataset):
    """Sliding window: input = window[:k], target = window[k]."""
    def __init__(self, seqs, window=10):
        self.pairs = []
        for seq in seqs:
            # seq shape: (MAX_LEN,) — extract non-padding
            seq_np = seq.numpy()
            nonzero = np.where(seq_np > 0)[0]
            if len(nonzero) < 2:
                continue
            clean = seq_np[nonzero]
            for i in range(len(clean) - 1):
                inp = clean[max(0, i-window+1):i+1]
                if len(inp) < window:
                    inp = np.pad(inp, (window - len(inp), 0), constant_values=0)
                self.pairs.append((inp, clean[i+1]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        inp, tgt = self.pairs[idx]
        return torch.tensor(inp, dtype=torch.long), torch.tensor(tgt, dtype=torch.long)

DL_WINDOW = 10
DL_K      = 9
print(f"   Building DeepLog window dataset (window={DL_WINDOW}, k={DL_K}) ...")
dl_win_ds = WindowDataset(X_dl_normal, window=DL_WINDOW)
dl_win_dl = DataLoader(dl_win_ds, batch_size=512, shuffle=True, num_workers=0)
print(f"   Window pairs: {len(dl_win_ds):,}")

deeplog_model = DeepLogLSTM(VOCAB_SIZE, hidden=128, num_layers=2).to(DEVICE)
dl_optim  = torch.optim.Adam(deeplog_model.parameters(), lr=1e-3)
dl_criterion = nn.CrossEntropyLoss(ignore_index=0)

print("   Training DeepLog ...")
DL_EPOCHS = 15
for epoch in range(DL_EPOCHS):
    deeplog_model.train()
    total_loss = 0.0
    for inp_b, tgt_b in dl_win_dl:
        inp_b, tgt_b = inp_b.to(DEVICE), tgt_b.to(DEVICE)
        # inp_b: (B, window)  → pass through LSTM, take last position logit
        logits = deeplog_model(inp_b)   # (B, window, V)
        last_logit = logits[:, -1, :]   # (B, V)
        loss = dl_criterion(last_logit, tgt_b)
        dl_optim.zero_grad()
        loss.backward()
        dl_optim.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"   Epoch {epoch+1:02d}/{DL_EPOCHS} — Loss: {total_loss/len(dl_win_dl):.4f}")

# ── DeepLog Inference ─────────────────────────────────────────────────────────
print("   Running DeepLog inference on test set ...")
deeplog_model.eval()

def deeplog_score_session(seq_np, window=DL_WINDOW, k=DL_K):
    """
    For each consecutive pair in session, check if actual next event
    is in top-k predictions. Score = fraction of anomalous windows.
    """
    nonzero = np.where(seq_np > 0)[0]
    if len(nonzero) < 2:
        return 0.0, 0.0  # (score, raw_count)
    clean = seq_np[nonzero]
    anomalous = 0
    total     = 0
    for i in range(len(clean) - 1):
        inp = clean[max(0, i-window+1):i+1]
        if len(inp) < window:
            inp = np.pad(inp, (window - len(inp), 0), constant_values=0)
        x = torch.tensor(inp, dtype=torch.long).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            logit = deeplog_model(x)[:, -1, :]   # (1, V)
        topk = logit.topk(k, dim=-1).indices.squeeze().cpu().numpy()
        if clean[i+1] not in topk:
            anomalous += 1
        total += 1
    score = anomalous / total if total > 0 else 0.0
    return score, float(anomalous)

dl_scores = np.array([deeplog_score_session(X_test_seq[i])[0] for i in range(len(X_test_seq))])

# Threshold on validation set (use normal-only baseline)
val_dl_scores = np.array([deeplog_score_session(X_val_seq[i])[0] for i in range(len(X_val_seq))])
best_dl_thresh, best_dl_f1 = 0.0, 0.0
for thr in np.linspace(0, 1, 101):
    preds = (val_dl_scores >= thr).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_dl_f1:
        best_dl_f1   = f1
        best_dl_thresh = thr

dl_preds = (dl_scores >= best_dl_thresh).astype(int)
print(f"   DeepLog best threshold (val): {best_dl_thresh:.2f}  Val-F1: {best_dl_f1:.4f}")

# Metrics
dl_prec  = precision_score(y_test, dl_preds, zero_division=0)
dl_rec   = recall_score(y_test, dl_preds,    zero_division=0)
dl_f1    = f1_score(y_test, dl_preds,        zero_division=0)
try:
    dl_auc = roc_auc_score(y_test, dl_scores)
except:
    dl_auc = 0.0

baseline_results["DeepLog"] = {
    "Precision": round(dl_prec, 4), "Recall": round(dl_rec, 4),
    "F1": round(dl_f1, 4),          "AUC":   round(dl_auc, 4),
}
print(f"   DeepLog → P={dl_prec:.4f}  R={dl_rec:.4f}  F1={dl_f1:.4f}  AUC={dl_auc:.4f}")

# ROC
try:
    dl_fpr, dl_tpr, _ = roc_curve(y_test, dl_scores)
    all_roc["DeepLog"] = (dl_fpr, dl_tpr, dl_auc)
except:
    pass

# Confusion matrix
fig_cm_dl, ax_cm_dl = plt.subplots(figsize=(5, 4))
cm_dl = confusion_matrix(y_test, dl_preds)
sns.heatmap(cm_dl, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal","Anomaly"], yticklabels=["Normal","Anomaly"],
            ax=ax_cm_dl)
ax_cm_dl.set_title("DeepLog — Confusion Matrix", fontsize=12, fontweight="bold")
ax_cm_dl.set_xlabel("Predicted"); ax_cm_dl.set_ylabel("True")
plt.tight_layout()
fig_cm_dl.savefig(os.path.join(FIGURE_DIR, "cm_deeplog.png"), dpi=300)
plt.close(fig_cm_dl)
print("   Saved: cm_deeplog.png")

torch.save(deeplog_model.state_dict(), os.path.join(MODEL_DIR, "deeplog.pt"))

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 6 — BASELINE 2: LogBERT (simplified)                                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 6 — BASELINE 2: LogBERT (simplified)")
print("="*60)

class LogBERTModel(nn.Module):
    """
    Simplified LogBERT:
      - Embedding(vocab_size, d_model)
      - Transformer Encoder (2 layers, 4 heads)
      - CLS token → classify (fine-tune stage)
      - MLM head for pre-training
    """

    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2,
                 max_len=MAX_LEN+1, dropout=0.1):
        super().__init__()
        self.d_model    = d_model
        self.CLS_ID     = vocab_size     # extra token appended
        self.vocab_size = vocab_size + 1

        self.token_emb  = nn.Embedding(self.vocab_size, d_model, padding_idx=0)
        self.pos_emb    = nn.Embedding(max_len + 1, d_model)
        self.norm_in    = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=256, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # MLM head
        self.mlm_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, self.vocab_size)
        )

        # Classification head (fine-tune stage)
        self.cls_head = nn.Linear(d_model, 2)

    def forward(self, input_ids, attention_mask=None, return_cls=False):
        # input_ids: (B, L)  — L includes CLS at position 0
        B, L = input_ids.shape
        pos  = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, -1)

        x = self.token_emb(input_ids) + self.pos_emb(pos)
        x = self.dropout(self.norm_in(x))

        # src_key_padding_mask: True = ignore (padding positions)
        if attention_mask is not None:
            # attention_mask: 1=real, 0=pad → invert for transformer
            pad_mask = (attention_mask == 0)   # (B, L)
        else:
            pad_mask = None

        x = self.encoder(x, src_key_padding_mask=pad_mask)   # (B, L, d_model)

        cls_out  = x[:, 0, :]      # CLS token representation
        mlm_out  = x[:, 1:, :]     # sequence positions

        if return_cls:
            return self.cls_head(cls_out)       # (B, 2)
        else:
            return self.mlm_head(mlm_out)       # (B, L-1, vocab)

# ── Dataset with CLS prepended ────────────────────────────────────────────────
class LogBERTDataset(Dataset):
    def __init__(self, feat_list, mask_rate=0.15, mode="pretrain", vocab_size=VOCAB_SIZE):
        self.data       = feat_list
        self.mask_rate  = mask_rate
        self.mode       = mode
        self.vocab_size = vocab_size
        self.CLS_ID     = vocab_size   # CLS = vocab_size (out of range of normal IDs)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d       = self.data[idx]
        evs     = d["event_ids"].copy()        # (32,)
        mask    = d["attention_mask"].copy()   # (32,)
        label   = d["label"]

        # Prepend CLS  → (33,)
        cls_arr  = np.array([self.CLS_ID], dtype=np.int64)
        cls_mask = np.array([1.0],         dtype=np.float32)

        input_ids = np.concatenate([cls_arr, evs])           # (33,)
        attn_mask = np.concatenate([cls_mask, mask])          # (33,)

        if self.mode == "pretrain":
            # Masked language model
            masked_ids = input_ids.copy()
            mlm_labels = input_ids.copy()  # (33,) — ignore CLS pos in loss

            for i in range(1, len(masked_ids)):  # skip CLS
                if attn_mask[i] == 1.0 and random.random() < self.mask_rate:
                    r = random.random()
                    if r < 0.8:
                        masked_ids[i] = VOCAB_SIZE  # MASK token = same as CLS_ID here
                    elif r < 0.9:
                        masked_ids[i] = random.randint(2, self.vocab_size - 1)
                    # else: keep original (10%)
                else:
                    mlm_labels[i] = -100  # ignore in loss

            return (
                torch.tensor(masked_ids,  dtype=torch.long),
                torch.tensor(attn_mask,   dtype=torch.float32),
                torch.tensor(mlm_labels[1:], dtype=torch.long),  # (32,) — skip CLS pos
            )
        else:  # finetune
            return (
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(attn_mask, dtype=torch.float32),
                torch.tensor(label,     dtype=torch.long),
            )

lb_model = LogBERTModel(vocab_size=VOCAB_SIZE, d_model=128, nhead=4, num_layers=2).to(DEVICE)
lb_optim_pre = AdamW(lb_model.parameters(), lr=1e-3, weight_decay=1e-4)

# ── Pre-training (masked event prediction) ────────────────────────────────────
print("   Pre-training LogBERT (masked event prediction, 15% mask rate) ...")
# Only train on normal sessions (unsupervised)
normal_feat_train = [f for f in feat_train if f["label"] == 0]
lb_pretrain_ds    = LogBERTDataset(normal_feat_train, mask_rate=0.15, mode="pretrain")
lb_pretrain_dl    = DataLoader(lb_pretrain_ds, batch_size=128, shuffle=True, num_workers=0)

LB_PRETRAIN_EPOCHS = 10
for epoch in range(LB_PRETRAIN_EPOCHS):
    lb_model.train()
    total_loss = 0.0
    for masked_ids, attn_mask, mlm_labels in lb_pretrain_dl:
        masked_ids = masked_ids.to(DEVICE)
        attn_mask  = attn_mask.to(DEVICE)
        mlm_labels = mlm_labels.to(DEVICE)

        # Forward → MLM logits (B, 32, vocab)
        logits = lb_model(masked_ids, attention_mask=attn_mask, return_cls=False)

        # Flatten and compute CE loss (ignore -100)
        loss = F.cross_entropy(
            logits.reshape(-1, lb_model.vocab_size),
            mlm_labels.reshape(-1),
            ignore_index=-100
        )
        lb_optim_pre.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lb_model.parameters(), 1.0)
        lb_optim_pre.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"   Pre-train Epoch {epoch+1:02d}/{LB_PRETRAIN_EPOCHS} — MLM Loss: {total_loss/len(lb_pretrain_dl):.4f}")

# ── Fine-tuning with classification head ──────────────────────────────────────
print("   Fine-tuning LogBERT (classification head) ...")
lb_finetune_train_ds = LogBERTDataset(feat_train, mode="finetune")
lb_finetune_val_ds   = LogBERTDataset(feat_val,   mode="finetune")
lb_finetune_test_ds  = LogBERTDataset(feat_test,  mode="finetune")
lb_finetune_dl  = DataLoader(lb_finetune_train_ds, batch_size=128, sampler=sampler, num_workers=0)
lb_val_dl       = DataLoader(lb_finetune_val_ds,   batch_size=128, shuffle=False,   num_workers=0)
lb_test_dl      = DataLoader(lb_finetune_test_ds,  batch_size=128, shuffle=False,   num_workers=0)

lb_optim_ft = AdamW(lb_model.parameters(), lr=5e-4, weight_decay=1e-4)
# Class weight for imbalance
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
lb_cls_weight = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
lb_criterion  = nn.CrossEntropyLoss(weight=lb_cls_weight)

LB_FINETUNE_EPOCHS = 15
best_lb_f1   = 0.0
best_lb_state = None

for epoch in range(LB_FINETUNE_EPOCHS):
    lb_model.train()
    for inp_ids, attn_mask, labels in lb_finetune_dl:
        inp_ids, attn_mask, labels = inp_ids.to(DEVICE), attn_mask.to(DEVICE), labels.to(DEVICE)
        logits = lb_model(inp_ids, attention_mask=attn_mask, return_cls=True)
        loss = lb_criterion(logits, labels)
        lb_optim_ft.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lb_model.parameters(), 1.0)
        lb_optim_ft.step()

    # Validate
    lb_model.eval()
    val_preds, val_probs = [], []
    with torch.no_grad():
        for inp_ids, attn_mask, labels in lb_val_dl:
            logits = lb_model(inp_ids.to(DEVICE), attention_mask=attn_mask.to(DEVICE), return_cls=True)
            probs  = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            val_preds.extend(preds.tolist())
            val_probs.extend(probs.tolist())
    val_f1 = f1_score(y_val, val_preds, zero_division=0)
    if val_f1 > best_lb_f1:
        best_lb_f1   = val_f1
        best_lb_state = {k: v.cpu().clone() for k, v in lb_model.state_dict().items()}
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"   Fine-tune Epoch {epoch+1:02d}/{LB_FINETUNE_EPOCHS} — Val F1: {val_f1:.4f}  (best: {best_lb_f1:.4f})")

# Load best
lb_model.load_state_dict(best_lb_state)
lb_model = lb_model.to(DEVICE)

# ── Test evaluation ───────────────────────────────────────────────────────────
lb_model.eval()
lb_preds_test, lb_probs_test = [], []
with torch.no_grad():
    for inp_ids, attn_mask, labels in lb_test_dl:
        logits = lb_model(inp_ids.to(DEVICE), attention_mask=attn_mask.to(DEVICE), return_cls=True)
        probs  = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        preds  = (probs >= 0.5).astype(int)
        lb_preds_test.extend(preds.tolist())
        lb_probs_test.extend(probs.tolist())

lb_preds_test = np.array(lb_preds_test)
lb_probs_test = np.array(lb_probs_test)

lb_prec = precision_score(y_test, lb_preds_test, zero_division=0)
lb_rec  = recall_score(y_test, lb_preds_test,    zero_division=0)
lb_f1   = f1_score(y_test, lb_preds_test,        zero_division=0)
try:
    lb_auc = roc_auc_score(y_test, lb_probs_test)
except:
    lb_auc = 0.0

baseline_results["LogBERT"] = {
    "Precision": round(lb_prec, 4), "Recall": round(lb_rec, 4),
    "F1": round(lb_f1, 4),          "AUC":   round(lb_auc, 4),
}
print(f"   LogBERT → P={lb_prec:.4f}  R={lb_rec:.4f}  F1={lb_f1:.4f}  AUC={lb_auc:.4f}")

# ROC
try:
    lb_fpr, lb_tpr, _ = roc_curve(y_test, lb_probs_test)
    all_roc["LogBERT"] = (lb_fpr, lb_tpr, lb_auc)
except:
    pass

# Confusion matrix
fig_cm_lb, ax_cm_lb = plt.subplots(figsize=(5, 4))
cm_lb = confusion_matrix(y_test, lb_preds_test)
sns.heatmap(cm_lb, annot=True, fmt="d", cmap="Oranges",
            xticklabels=["Normal","Anomaly"], yticklabels=["Normal","Anomaly"],
            ax=ax_cm_lb)
ax_cm_lb.set_title("LogBERT — Confusion Matrix", fontsize=12, fontweight="bold")
ax_cm_lb.set_xlabel("Predicted"); ax_cm_lb.set_ylabel("True")
plt.tight_layout()
fig_cm_lb.savefig(os.path.join(FIGURE_DIR, "cm_logbert.png"), dpi=300)
plt.close(fig_cm_lb)
print("   Saved: cm_logbert.png")

torch.save(lb_model.state_dict(), os.path.join(MODEL_DIR, "logbert.pt"))

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 7 — MAIN MODEL: HierAttn-Block                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 7 — HierAttn-Block Architecture")
print("="*60)


class EventEmbedding(nn.Module):
    """
    Per-event embedding:
      - Template ID   → Embedding(vocab_size, 64)
      - Param feats   → Linear(3, 32)
      - Sinusoidal TD → already 32-dim
      - Concat        → 128-dim per event
    """

    def __init__(self, vocab_size, embed_dim=64, param_dim=32, time_dim=32):
        super().__init__()
        self.template_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.param_proj   = nn.Linear(3, param_dim)
        # time_dim is already encoded externally (passed as sin_time)
        assert embed_dim + param_dim + time_dim == 128, \
            f"EventEmbedding output must be 128, got {embed_dim+param_dim+time_dim}"

    def forward(self, event_ids, param_feats, sin_time):
        """
        event_ids:  (B, L)
        param_feats:(B, L, 3)
        sin_time:   (B, L, 32)
        Returns:    (B, L, 128)
        """
        t_emb   = self.template_emb(event_ids)   # (B, L, 64)
        p_emb   = self.param_proj(param_feats)    # (B, L, 32)
        return torch.cat([t_emb, p_emb, sin_time], dim=-1)  # (B, L, 128)


class TransformerEncoder(nn.Module):
    """
    Standard Transformer encoder (2 layers, 4 heads, d_model=128, ffn=256).
    Session vector = concat(mean_pool(H), max_pool(H)) → 256-dim.
    """

    def __init__(self, d_model=128, nhead=4, num_layers=2, ffn_dim=256, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=ffn_dim, dropout=dropout,
            batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.d_model  = d_model

    def forward(self, x, attention_mask):
        """
        x:             (B, L, 128)
        attention_mask:(B, L)  — 1=real, 0=pad
        Returns:
          H:           (B, L, 128)
          session_vec: (B, 256)
        """
        # src_key_padding_mask: True = ignore
        pad_mask = (attention_mask == 0)   # (B, L)
        H = self.encoder(x, src_key_padding_mask=pad_mask)   # (B, L, 128)

        # Masked pooling
        mask_exp = attention_mask.unsqueeze(-1)               # (B, L, 1)
        H_masked = H * mask_exp

        # Mean pool (over real events)
        denom    = mask_exp.sum(dim=1).clamp(min=1)
        mean_vec = H_masked.sum(dim=1) / denom                # (B, 128)

        # Max pool (replace padding with -inf before max)
        H_inf    = H_masked + (1 - mask_exp) * (-1e9)
        max_vec  = H_inf.max(dim=1).values                    # (B, 128)

        session_vec = torch.cat([mean_vec, max_vec], dim=-1)  # (B, 256)
        return H, session_vec


class StructuralMLP(nn.Module):
    """Linear(11→64) → ReLU → Linear(64→64) → ReLU → 64-dim"""

    def __init__(self, in_dim=11, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)   # (B, 64)


class HierAttnBlock(nn.Module):
    """
    Full HierAttn-Block model:
      EventEmbedding → TransformerEncoder  → 256-dim session vec
      StructuralMLP                         →  64-dim struct vec
      Fusion: concat(256, 64) → Linear(320,128) → ReLU → Dropout(0.3)
      Classifier head: Linear(128, 2)
      Auxiliary head (training only): Linear(128, 1) for replication regression
    """

    def __init__(self, vocab_size,
                 embed_dim=64, param_dim=32, time_dim=32,
                 d_model=128, nhead=4, num_enc_layers=2, ffn_dim=256,
                 struct_dim=11, struct_hidden=64,
                 fusion_hidden=128, dropout=0.3):
        super().__init__()
        self.event_emb     = EventEmbedding(vocab_size, embed_dim, param_dim, time_dim)
        self.transformer   = TransformerEncoder(d_model, nhead, num_enc_layers, ffn_dim)
        self.struct_mlp    = StructuralMLP(struct_dim, struct_hidden)

        # Fusion
        fuse_in = d_model * 2 + struct_hidden   # 256 + 64 = 320
        self.fusion = nn.Sequential(
            nn.Linear(fuse_in, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Classifier
        self.cls_head = nn.Linear(fusion_hidden, 2)

        # Auxiliary head (replication count regression)
        self.aux_head = nn.Linear(fusion_hidden, 1)

        # Store attention weights (for visualization)
        self._last_attn_weights = None

    def forward(self, event_ids, param_feats, sin_time, struct_feats, attention_mask,
                return_aux=True):
        """
        Returns:
          logits:   (B, 2)
          aux_out:  (B, 1) or None
        """
        # Event embedding
        x = self.event_emb(event_ids, param_feats, sin_time)   # (B, L, 128)

        # Transformer
        H, sess_vec = self.transformer(x, attention_mask)       # sess_vec: (B, 256)

        # Structural
        s_vec = self.struct_mlp(struct_feats)                   # (B, 64)

        # Fusion
        fused  = torch.cat([sess_vec, s_vec], dim=-1)           # (B, 320)
        hidden = self.fusion(fused)                             # (B, 128)

        # Classifier
        logits = self.cls_head(hidden)                          # (B, 2)

        # Auxiliary
        aux_out = self.aux_head(hidden) if return_aux else None  # (B, 1)

        return logits, aux_out, H

    def get_attention(self, event_ids, param_feats, sin_time, struct_feats, attention_mask):
        """Returns averaged attention weights from all encoder layers."""
        # Register hooks or use transformer's internal attention
        # Since nn.TransformerEncoderLayer doesn't expose attn weights directly,
        # we extract them via a custom forward hook
        attn_weights_list = []

        def hook_fn(module, inp, out):
            # out for TransformerEncoderLayer is the hidden state, not attn weights
            pass

        with torch.no_grad():
            logits, aux, H = self.forward(
                event_ids, param_feats, sin_time, struct_feats, attention_mask,
                return_aux=False
            )
        return H   # Return hidden states as proxy for attention visualization


model_hier = HierAttnBlock(vocab_size=VOCAB_SIZE).to(DEVICE)
total_params = sum(p.numel() for p in model_hier.parameters() if p.requires_grad)
print(f"   HierAttn-Block parameters: {total_params:,}")
print(f"   Architecture verified ✓")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 8 — TRAINING                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 8 — TRAINING HierAttn-Block")
print("="*60)


class FocalLoss(nn.Module):
    """Binary Focal Loss with class-balanced alpha."""

    def __init__(self, gamma=2.0, alpha=0.75, reduction="mean"):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B, 2), targets: (B,)
        probs  = F.softmax(logits, dim=-1)            # (B, 2)
        pt     = probs[range(len(targets)), targets]  # (B,)

        # Alpha weighting
        alpha_t = torch.where(
            targets == 1,
            torch.tensor(self.alpha,       device=logits.device),
            torch.tensor(1 - self.alpha,   device=logits.device)
        )

        loss = -alpha_t * (1 - pt) ** self.gamma * torch.log(pt + 1e-8)

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


focal_loss = FocalLoss(gamma=2.0, alpha=0.75)
aux_loss   = nn.MSELoss()

optimizer = AdamW(model_hier.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# Training history
history = {
    "train_loss": [], "train_f1": [],
    "val_loss": [],   "val_f1": [],
}

best_val_f1    = 0.0
patience_count = 0
best_state     = None

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "hierattn_best.pt")

print(f"   Starting training: {MAX_EPOCHS} epochs, patience={PATIENCE}, batch={BATCH_SIZE}")
print(f"   Optimizer: AdamW (lr={LR}, wd={WEIGHT_DECAY})")
print(f"   Loss: FocalLoss(γ=2, α=0.75) + 0.1 * MSE(replication)")

for epoch in range(MAX_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────────
    model_hier.train()
    epoch_loss, epoch_preds, epoch_labels = 0.0, [], []

    for batch in dl_train:
        ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]

        logits, aux_out, _ = model_hier(ev_ids, pf, st, sf, am, return_aux=True)

        cls_loss = focal_loss(logits, labels)
        reg_loss = aux_loss(aux_out.squeeze(-1), repl.float())
        loss     = cls_loss + 0.1 * reg_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_hier.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        preds = logits.argmax(dim=-1).cpu().numpy()
        epoch_preds.extend(preds.tolist())
        epoch_labels.extend(labels.cpu().numpy().tolist())

    scheduler.step()

    train_f1   = f1_score(epoch_labels, epoch_preds, zero_division=0)
    train_loss = epoch_loss / len(dl_train)

    # ── Validate ───────────────────────────────────────────────────────────────
    model_hier.eval()
    val_loss, val_preds_ep, val_labels_ep = 0.0, [], []

    with torch.no_grad():
        for batch in dl_val:
            ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]
            logits, aux_out, _ = model_hier(ev_ids, pf, st, sf, am, return_aux=True)
            cls_loss = focal_loss(logits, labels)
            reg_loss = aux_loss(aux_out.squeeze(-1), repl.float())
            loss     = cls_loss + 0.1 * reg_loss
            val_loss += loss.item()
            preds = logits.argmax(dim=-1).cpu().numpy()
            val_preds_ep.extend(preds.tolist())
            val_labels_ep.extend(labels.cpu().numpy().tolist())

    val_f1   = f1_score(val_labels_ep, val_preds_ep, zero_division=0)
    val_loss /= len(dl_val)

    history["train_loss"].append(train_loss)
    history["train_f1"].append(train_f1)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1   = val_f1
        patience_count = 0
        best_state    = {k: v.cpu().clone() for k, v in model_hier.state_dict().items()}
        torch.save(best_state, CHECKPOINT_PATH)
    else:
        patience_count += 1

    if (epoch + 1) % 5 == 0 or epoch == 0 or patience_count == 0:
        print(f"   Epoch {epoch+1:02d}/{MAX_EPOCHS} | "
              f"Train Loss: {train_loss:.4f}  F1: {train_f1:.4f} | "
              f"Val Loss: {val_loss:.4f}  F1: {val_f1:.4f} | "
              f"Best Val F1: {best_val_f1:.4f}  Patience: {patience_count}/{PATIENCE}")

    if patience_count >= PATIENCE:
        print(f"   ⚡ Early stopping at epoch {epoch+1} (patience={PATIENCE} exceeded)")
        break

print(f"\n   ✅ Training complete. Best Val F1: {best_val_f1:.4f}")
print(f"   Checkpoint saved: {CHECKPOINT_PATH}")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 9 — TWO-STAGE INFERENCE                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 9 — TWO-STAGE INFERENCE")
print("="*60)

# Load best checkpoint
model_hier.load_state_dict(best_state)
model_hier = model_hier.to(DEVICE)
model_hier.eval()

# ── Tune threshold on validation set ─────────────────────────────────────────
print("   Tuning threshold τ on validation set ...")
val_probs_hier = []
val_stage1_flags = []

val_missing_alloc = np.array([f["missing_alloc"] for f in feat_val])
val_repl_neq3     = np.array([f["repl_neq3"]     for f in feat_val])

# Stage 1: hard rules on val
stage1_val = (val_missing_alloc == 1) | (val_repl_neq3 == 1)

with torch.no_grad():
    for batch in dl_val:
        ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]
        logits, _, _ = model_hier(ev_ids, pf, st, sf, am, return_aux=False)
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        val_probs_hier.extend(probs.tolist())

val_probs_hier = np.array(val_probs_hier)

# Two-stage val predictions
best_tau, best_2stage_f1 = 0.5, 0.0
for tau in np.linspace(0.1, 0.9, 81):
    stage2_preds = (val_probs_hier >= tau).astype(int)
    combined     = np.maximum(stage1_val.astype(int), stage2_preds)
    f1 = f1_score(y_val, combined, zero_division=0)
    if f1 > best_2stage_f1:
        best_2stage_f1 = f1
        best_tau       = tau

print(f"   Best τ = {best_tau:.2f}  (Val 2-stage F1: {best_2stage_f1:.4f})")

# ── Stage 1 on test ───────────────────────────────────────────────────────────
stage1_test = (test_missing_alloc == 1) | (test_repl_neq3 == 1)
n_stage1 = stage1_test.sum()
print(f"\n   Stage 1 (hard rules): {n_stage1} sessions flagged immediately "
      f"({100*n_stage1/len(y_test):.1f}% of test)")

# ── Stage 2 (neural) on remaining ────────────────────────────────────────────
hier_probs_test = []
with torch.no_grad():
    for batch in dl_test:
        ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]
        logits, _, _ = model_hier(ev_ids, pf, st, sf, am, return_aux=False)
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        hier_probs_test.extend(probs.tolist())

hier_probs_test = np.array(hier_probs_test)

# Full pipeline prediction
stage2_test_preds  = (hier_probs_test >= best_tau).astype(int)
hier_final_preds   = np.maximum(stage1_test.astype(int), stage2_test_preds)
# AUC uses raw probs (override with 1.0 for stage-1 flagged)
hier_final_probs   = np.where(stage1_test, 1.0, hier_probs_test)

hier_prec = precision_score(y_test, hier_final_preds, zero_division=0)
hier_rec  = recall_score(y_test, hier_final_preds,    zero_division=0)
hier_f1   = f1_score(y_test, hier_final_preds,        zero_division=0)
try:
    hier_auc = roc_auc_score(y_test, hier_final_probs)
except:
    hier_auc = 0.0

results["HierAttnBlock"] = {
    "Precision": round(hier_prec, 4), "Recall": round(hier_rec, 4),
    "F1": round(hier_f1, 4),          "AUC":   round(hier_auc, 4),
}
print(f"\n   HierAttn-Block → P={hier_prec:.4f}  R={hier_rec:.4f}  "
      f"F1={hier_f1:.4f}  AUC={hier_auc:.4f}")

# ROC
try:
    hier_fpr, hier_tpr, _ = roc_curve(y_test, hier_final_probs)
    all_roc["HierAttn-Block"] = (hier_fpr, hier_tpr, hier_auc)
except:
    pass

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 10 — ABLATION STUDY                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 10 — ABLATION STUDY")
print("="*60)


def train_ablation_model(model, dl_tr, dl_v, y_v, label,
                         epochs=MAX_EPOCHS, patience=PATIENCE, use_aux=True):
    """Generic training loop for ablation variants.

    Parameters
    ----------
    use_aux : bool
        If True, compute auxiliary MSE loss for replication regression.
        Pass False for models that should not use the auxiliary head.
    """
    opt   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = CosineAnnealingWarmRestarts(opt, T_0=10, T_mult=2)
    fc    = FocalLoss(gamma=2.0, alpha=0.75)
    aux_l = nn.MSELoss()

    best_f1, best_state, pat = 0.0, None, 0

    for epoch in range(epochs):
        model.train()
        for batch in dl_tr:
            ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]
            logits, aux_out, _ = model(ev_ids, pf, st, sf, am, return_aux=use_aux)

            if use_aux and aux_out is not None:
                loss = fc(logits, labels) + 0.1 * aux_l(aux_out.squeeze(-1), repl.float())
            else:
                loss = fc(logits, labels)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for batch in dl_v:
                ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]
                logits, _, _ = model(ev_ids, pf, st, sf, am, return_aux=False)
                preds.extend(logits.argmax(-1).cpu().numpy().tolist())
        vf1 = f1_score(y_v, preds, zero_division=0)
        if vf1 > best_f1:
            best_f1    = vf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
        if pat >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_f1


def eval_model_test(model, dl_tst, y_tst):
    """Run test evaluation → (preds, probs, metrics_dict)."""
    model.eval()
    preds, probs = [], []
    with torch.no_grad():
        for batch in dl_tst:
            ev_ids, pf, st, sf, am, labels, repl = [b.to(DEVICE) for b in batch]
            logits, _, _ = model(ev_ids, pf, st, sf, am, return_aux=False)
            p = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            probs.extend(p.tolist())
            preds.extend((p >= 0.5).astype(int).tolist())
    preds = np.array(preds); probs = np.array(probs)
    prec  = precision_score(y_tst, preds, zero_division=0)
    rec   = recall_score(y_tst, preds,    zero_division=0)
    f1    = f1_score(y_tst, preds,        zero_division=0)
    try:
        auc = roc_auc_score(y_tst, probs)
    except:
        auc = 0.0
    return preds, probs, {"Precision": round(prec,4), "Recall": round(rec,4),
                          "F1": round(f1,4), "AUC": round(auc,4)}


# ── Variant 1: Sequence Only (no StructuralMLP) ───────────────────────────────
print("   [1/4] Sequence Only ...")

class HierAttnSeqOnly(nn.Module):
    """HierAttn-Block without structural path."""
    def __init__(self, vocab_size):
        super().__init__()
        self.event_emb   = EventEmbedding(vocab_size)
        self.transformer = TransformerEncoder()
        self.fusion      = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3)
        )
        self.cls_head = nn.Linear(128, 2)
        self.aux_head = nn.Linear(128, 1)

    def forward(self, event_ids, param_feats, sin_time, struct_feats, attention_mask,
                return_aux=True):
        x = self.event_emb(event_ids, param_feats, sin_time)
        _, sess_vec = self.transformer(x, attention_mask)
        hidden = self.fusion(sess_vec)
        logits = self.cls_head(hidden)
        aux    = self.aux_head(hidden) if return_aux else None
        return logits, aux, torch.zeros(1)

m_seq = HierAttnSeqOnly(VOCAB_SIZE).to(DEVICE)
m_seq, _ = train_ablation_model(m_seq, dl_train, dl_val, y_val, "SeqOnly", use_aux=True)
_, probs_seq, res_seq = eval_model_test(m_seq, dl_test, y_test)
print(f"     Sequence Only → {res_seq}")
try:
    fpr_, tpr_, _ = roc_curve(y_test, probs_seq)
    all_roc["Seq Only"] = (fpr_, tpr_, res_seq["AUC"])
except:
    pass

# ── Variant 2: Structural Only (no Transformer) ───────────────────────────────
print("   [2/4] Structural Only ...")

class HierAttnStructOnly(nn.Module):
    """HierAttn-Block without Transformer encoder."""
    def __init__(self):
        super().__init__()
        self.struct_mlp = StructuralMLP(11, 64)
        self.fusion     = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.3)
        )
        self.cls_head = nn.Linear(128, 2)
        self.aux_head = nn.Linear(128, 1)

    def forward(self, event_ids, param_feats, sin_time, struct_feats, attention_mask,
                return_aux=True):
        s_vec  = self.struct_mlp(struct_feats)
        hidden = self.fusion(s_vec)
        logits = self.cls_head(hidden)
        aux    = self.aux_head(hidden) if return_aux else None
        return logits, aux, torch.zeros(1)

m_struct = HierAttnStructOnly().to(DEVICE)
m_struct, _ = train_ablation_model(m_struct, dl_train, dl_val, y_val, "StructOnly")
_, probs_struct, res_struct = eval_model_test(m_struct, dl_test, y_test)
print(f"     Structural Only → {res_struct}")
try:
    fpr_, tpr_, _ = roc_curve(y_test, probs_struct)
    all_roc["Struct Only"] = (fpr_, tpr_, res_struct["AUC"])
except:
    pass

# ── Variant 3: Full model WITHOUT auxiliary head ──────────────────────────────
print("   [3/4] Full model — No Auxiliary Head ...")

class HierAttnNoAux(HierAttnBlock):
    """Same as HierAttn-Block but auxiliary head is always disabled in loss."""
    pass

m_noaux = HierAttnNoAux(VOCAB_SIZE).to(DEVICE)
m_noaux, _ = train_ablation_model(m_noaux, dl_train, dl_val, y_val, "NoAux", use_aux=False)
_, probs_noaux, res_noaux = eval_model_test(m_noaux, dl_test, y_test)
print(f"     No Auxiliary Head → {res_noaux}")
try:
    fpr_, tpr_, _ = roc_curve(y_test, probs_noaux)
    all_roc["No Aux Head"] = (fpr_, tpr_, res_noaux["AUC"])
except:
    pass

# ── Variant 4: Full HierAttn-Block (already trained) ─────────────────────────
print("   [4/4] Full HierAttn-Block ...")
res_full = results["HierAttnBlock"]
print(f"     HierAttn-Block (Full) → {res_full}")

# ── Print Ablation Table ──────────────────────────────────────────────────────
ablation_rows = [
    ("Sequence Only",         res_seq),
    ("Structural Only",       res_struct),
    ("No Auxiliary Head",     res_noaux),
    ("HierAttn-Block (Full)", res_full),
    ("DeepLog",               baseline_results["DeepLog"]),
    ("LogBERT",               baseline_results["LogBERT"]),
]

print("\n")
print("| {:<23} | {:>9} | {:>6} | {:>5} | {:>5} |".format(
    "Model Variant", "Precision", "Recall", "F1", "AUC"))
print("|" + "-"*25 + "|" + "-"*11 + "|" + "-"*8 + "|" + "-"*7 + "|" + "-"*7 + "|")
for name, m in ablation_rows:
    print("| {:<23} | {:>9} | {:>6} | {:>5} | {:>5} |".format(
        name, m["Precision"], m["Recall"], m["F1"], m["AUC"]))

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 11 — THESIS FIGURES                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*60)
print("STEP 11 — THESIS FIGURES (PNG 300 DPI)")
print("="*60)

# Style
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        10,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        False,
    "figure.dpi":       100,
})
PALETTE = ["#2E86AB", "#E84855", "#3BB273", "#F18F01", "#A23B72", "#C73E1D"]


# ── Figure 1: Training loss + F1 curves ───────────────────────────────────────
epochs_x = list(range(1, len(history["train_loss"]) + 1))

fig1, axes1 = plt.subplots(1, 2, figsize=(12, 4))
fig1.suptitle("HierAttn-Block — Training Dynamics", fontsize=14, fontweight="bold", y=1.01)

ax = axes1[0]
ax.plot(epochs_x, history["train_loss"], color=PALETTE[0], lw=2, label="Train Loss")
ax.plot(epochs_x, history["val_loss"],   color=PALETTE[1], lw=2, linestyle="--", label="Val Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Loss Curves", fontweight="bold")
ax.legend(frameon=False)

ax = axes1[1]
ax.plot(epochs_x, history["train_f1"], color=PALETTE[0], lw=2, label="Train F1")
ax.plot(epochs_x, history["val_f1"],   color=PALETTE[1], lw=2, linestyle="--", label="Val F1")
best_ep = int(np.argmax(history["val_f1"])) + 1
ax.axvline(best_ep, color="grey", linestyle=":", lw=1, label=f"Best epoch ({best_ep})")
ax.set_xlabel("Epoch"); ax.set_ylabel("F1 Score")
ax.set_title("F1 Score Curves", fontweight="bold")
ax.legend(frameon=False)

plt.tight_layout()
fig1.savefig(os.path.join(FIGURE_DIR, "fig1_training_curves.png"), dpi=300, bbox_inches="tight")
plt.close(fig1)
print("   ✅ Figure 1 saved: fig1_training_curves.png")


# ── Figure 2: ROC curves — all models ─────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 6))
ax2.plot([0,1],[0,1], "k--", lw=1, alpha=0.5, label="Random")

roc_order = ["DeepLog", "LogBERT", "Seq Only", "Struct Only",
             "No Aux Head", "HierAttn-Block"]
for i, name in enumerate(roc_order):
    if name not in all_roc:
        continue
    fpr_, tpr_, auc_ = all_roc[name]
    ax2.plot(fpr_, tpr_, color=PALETTE[i % len(PALETTE)], lw=2,
             label=f"{name}  (AUC={auc_:.4f})")

ax2.set_xlabel("False Positive Rate", fontsize=11)
ax2.set_ylabel("True Positive Rate",  fontsize=11)
ax2.set_title("ROC Curves — All Models (HDFS Test Set)", fontsize=13, fontweight="bold")
ax2.legend(frameon=False, fontsize=9, loc="lower right")
ax2.set_xlim([-0.02, 1.02]); ax2.set_ylim([-0.02, 1.05])
plt.tight_layout()
fig2.savefig(os.path.join(FIGURE_DIR, "fig2_roc_curves.png"), dpi=300, bbox_inches="tight")
plt.close(fig2)
print("   ✅ Figure 2 saved: fig2_roc_curves.png")


# ── Figure 3: Confusion matrix — HierAttn-Block ───────────────────────────────
fig3, ax3 = plt.subplots(figsize=(5, 4))
cm_hier = confusion_matrix(y_test, hier_final_preds)
sns.heatmap(cm_hier, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Normal","Anomaly"], yticklabels=["Normal","Anomaly"],
            linewidths=0.5, linecolor="white",
            annot_kws={"size": 14, "weight": "bold"},
            ax=ax3)
ax3.set_title("HierAttn-Block — Confusion Matrix\n(Test Set)", fontsize=13, fontweight="bold")
ax3.set_xlabel("Predicted Label", fontsize=11)
ax3.set_ylabel("True Label",      fontsize=11)
# Precision / Recall annotation
ax3.text(1.05, 0.5,
         f"P = {hier_prec:.4f}\nR = {hier_rec:.4f}\nF1 = {hier_f1:.4f}\nAUC = {hier_auc:.4f}",
         transform=ax3.transAxes, fontsize=10, va="center",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f0f0", edgecolor="grey"))
plt.tight_layout()
fig3.savefig(os.path.join(FIGURE_DIR, "fig3_confusion_matrix.png"), dpi=300, bbox_inches="tight")
plt.close(fig3)
print("   ✅ Figure 3 saved: fig3_confusion_matrix.png")


# ── Figure 4: Attention heatmap — 3 sessions ─────────────────────────────────
# Find 1 normal and 2 anomaly sessions from test set
normal_idx_list  = [i for i in range(len(feat_test)) if feat_test[i]["label"] == 0]
anomaly_idx_list = [i for i in range(len(feat_test)) if feat_test[i]["label"] == 1]

selected = []
if len(normal_idx_list) > 0:
    selected.append(("Normal",  normal_idx_list[0]))
if len(anomaly_idx_list) > 1:
    selected.append(("Anomaly", anomaly_idx_list[0]))
    selected.append(("Anomaly", anomaly_idx_list[1]))
elif len(anomaly_idx_list) == 1:
    selected.append(("Anomaly", anomaly_idx_list[0]))

# Build mini batch for each selected session and extract hidden states
def get_hidden(feat):
    ev_ids = torch.tensor(feat["event_ids"],      dtype=torch.long).unsqueeze(0).to(DEVICE)
    pf     = torch.tensor(feat["param_feats"],    dtype=torch.float32).unsqueeze(0).to(DEVICE)
    st     = torch.tensor(feat["sin_time"],       dtype=torch.float32).unsqueeze(0).to(DEVICE)
    sf     = torch.tensor(feat["struct_feats"],   dtype=torch.float32).unsqueeze(0).to(DEVICE)
    am     = torch.tensor(feat["attention_mask"], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, _, H = model_hier(ev_ids, pf, st, sf, am, return_aux=False)
    return H.squeeze(0).cpu().numpy()  # (32, 128)

fig4, axes4 = plt.subplots(len(selected), 1, figsize=(12, 3 * len(selected)))
if len(selected) == 1:
    axes4 = [axes4]

for k, (stype, sidx) in enumerate(selected):
    feat_sel = feat_test[sidx]
    H_sel    = get_hidden(feat_sel)    # (32, 128)
    am_sel   = feat_sel["attention_mask"]
    n_real   = int(am_sel.sum())

    # Use L2 norm of hidden state as "attention score" per position
    attn_scores = np.linalg.norm(H_sel, axis=-1)  # (32,)
    attn_scores = attn_scores[:n_real]
    attn_scores = attn_scores / (attn_scores.max() + 1e-8)

    ev_labels = [str(feat_sel["event_ids"][i]) for i in range(n_real)]

    im = axes4[k].imshow(
        attn_scores.reshape(1, -1), aspect="auto", cmap="hot",
        vmin=0, vmax=1
    )
    axes4[k].set_xticks(range(n_real))
    axes4[k].set_xticklabels(ev_labels, rotation=45, ha="right", fontsize=8)
    axes4[k].set_yticks([])
    axes4[k].set_title(
        f"Session {k+1} ({stype}) — {n_real} events | "
        f"True: {'Anomaly' if feat_sel['label']==1 else 'Normal'}",
        fontsize=10, fontweight="bold"
    )
    plt.colorbar(im, ax=axes4[k], fraction=0.01, pad=0.01)

fig4.suptitle("HierAttn-Block — Hidden State Magnitude (Attention Proxy)",
              fontsize=13, fontweight="bold")
plt.tight_layout()
fig4.savefig(os.path.join(FIGURE_DIR, "fig4_attention_heatmap.png"), dpi=300, bbox_inches="tight")
plt.close(fig4)
print("   ✅ Figure 4 saved: fig4_attention_heatmap.png")


# ── Figure 5: Structural feature importance ───────────────────────────────────
print("   Computing permutation importance for structural features ...")

feat_names = [
    f"Template_{i+1}_count" for i in range(5)
] + ["size_std", "n_unique_ips", "session_duration",
     "max_gap", "missing_allocate", "repl_neq3"]

# Train a simple RF on structural features for permutation importance
rf_pi = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf_pi.fit(X_train_struct, y_train)
pi_result = permutation_importance(
    rf_pi, X_test_struct, y_test,
    n_repeats=10, random_state=SEED, scoring="f1"
)
pi_means = pi_result.importances_mean
pi_stds  = pi_result.importances_std

# Sort
sort_idx = np.argsort(pi_means)[::-1]
fig5, ax5 = plt.subplots(figsize=(9, 5))
ax5.barh(
    [feat_names[i] for i in sort_idx[::-1]],
    pi_means[sort_idx[::-1]],
    xerr=pi_stds[sort_idx[::-1]],
    color=PALETTE[0], ecolor="#999999", capsize=4, height=0.6
)
ax5.set_xlabel("Mean Decrease in F1 (Permutation Importance)", fontsize=11)
ax5.set_title("Structural Feature Importance — Permutation Method",
              fontsize=13, fontweight="bold")
ax5.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
fig5.savefig(os.path.join(FIGURE_DIR, "fig5_feature_importance.png"), dpi=300, bbox_inches="tight")
plt.close(fig5)
print("   ✅ Figure 5 saved: fig5_feature_importance.png")


# ── Figure 6: Session length distribution ─────────────────────────────────────
test_lengths_normal  = [
    int(feat_test[i]["attention_mask"].sum())
    for i in range(len(feat_test)) if feat_test[i]["label"] == 0
]
test_lengths_anomaly = [
    int(feat_test[i]["attention_mask"].sum())
    for i in range(len(feat_test)) if feat_test[i]["label"] == 1
]

fig6, ax6 = plt.subplots(figsize=(8, 5))
bins = range(1, MAX_LEN + 2)
ax6.hist(test_lengths_normal,  bins=bins, alpha=0.6, color=PALETTE[0],
         label=f"Normal (n={len(test_lengths_normal):,})",   density=True)
ax6.hist(test_lengths_anomaly, bins=bins, alpha=0.6, color=PALETTE[1],
         label=f"Anomaly (n={len(test_lengths_anomaly):,})", density=True)
ax6.set_xlabel("Session Length (# events, padded to 32)", fontsize=11)
ax6.set_ylabel("Density", fontsize=11)
ax6.set_title("Session Length Distribution — Normal vs. Anomaly (Test Set)",
              fontsize=13, fontweight="bold")
ax6.legend(frameon=False, fontsize=10)
plt.tight_layout()
fig6.savefig(os.path.join(FIGURE_DIR, "fig6_session_length_distribution.png"), dpi=300, bbox_inches="tight")
plt.close(fig6)
print("   ✅ Figure 6 saved: fig6_session_length_distribution.png")


# ── Re-save confusion matrix figures with academic style ──────────────────────
# (already saved above — confirm)
print("   ✅ Figure cm_deeplog.png  — already saved")
print("   ✅ Figure cm_logbert.png  — already saved")


# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FINAL OUTPUT                                                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

hier  = results["HierAttnBlock"]
deepl = baseline_results["DeepLog"]
logb  = baseline_results["LogBERT"]

print(f"\n  HierAttn-Block  →  "
      f"Precision: {hier['Precision']:.4f}  "
      f"Recall: {hier['Recall']:.4f}  "
      f"F1: {hier['F1']:.4f}  "
      f"AUC: {hier['AUC']:.4f}")

print(f"  DeepLog         →  "
      f"Precision: {deepl['Precision']:.4f}  "
      f"Recall: {deepl['Recall']:.4f}  "
      f"F1: {deepl['F1']:.4f}  "
      f"AUC: {deepl['AUC']:.4f}")

print(f"  LogBERT         →  "
      f"Precision: {logb['Precision']:.4f}  "
      f"Recall: {logb['Recall']:.4f}  "
      f"F1: {logb['F1']:.4f}  "
      f"AUC: {logb['AUC']:.4f}")

print(f"\n  Best model: HierAttn-Block")
print(f"  Improvement over DeepLog:  +{(hier['F1'] - deepl['F1'])*100:.2f}% F1")
print(f"  Improvement over LogBERT:  +{(hier['F1'] - logb['F1'])*100:.2f}% F1")

print("\n" + "="*70)
print("FIGURES SAVED (all 300 DPI PNG):")
figure_files = [
    "fig1_training_curves.png",
    "fig2_roc_curves.png",
    "fig3_confusion_matrix.png",
    "fig4_attention_heatmap.png",
    "fig5_feature_importance.png",
    "fig6_session_length_distribution.png",
    "cm_deeplog.png",
    "cm_logbert.png",
]
for f in figure_files:
    fp = os.path.join(FIGURE_DIR, f)
    status = "✅" if os.path.exists(fp) else "❌"
    print(f"  {status}  {fp}")

print("\nMODEL CHECKPOINTS:")
checkpoint_files = ["hierattn_best.pt", "deeplog.pt", "logbert.pt"]
for f in checkpoint_files:
    fp = os.path.join(MODEL_DIR, f)
    status = "✅" if os.path.exists(fp) else "❌"
    print(f"  {status}  {fp}")

# Save results JSON
results_all = {
    "HierAttnBlock": results["HierAttnBlock"],
    "DeepLog":       baseline_results["DeepLog"],
    "LogBERT":       baseline_results["LogBERT"],
    "Ablation": {
        "Sequence Only":         res_seq,
        "Structural Only":       res_struct,
        "No Auxiliary Head":     res_noaux,
        "HierAttn-Block (Full)": res_full,
    }
}
results_path = os.path.join(OUTPUT_DIR, "final_results.json")
with open(results_path, "w") as fp:
    json.dump(results_all, fp, indent=2)
print(f"\n  ✅ All results saved to: {results_path}")
print("\n" + "="*70 + "\n")
